<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Dailychallenge_J4_W2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv("superstore_dataset.csv", encoding='latin1')

# =====================================================
# DATA EXPLORATION
# =====================================================

print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Info:")
print(df.info())

print("\nSummary Statistics:")
display(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape:
(9994, 21)

Columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal C

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000



Missing Values:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [8]:
# =====================================================
# REMOVE DUPLICATES
# =====================================================

print("\nDuplicate Rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape After Removing Duplicates:", df.shape)

# =====================================================
# HANDLE MISSING VALUES
# =====================================================

if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

# =====================================================
# CONVERT DATE COLUMNS
# =====================================================

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])



Duplicate Rows: 0
Shape After Removing Duplicates: (9994, 21)


In [9]:
# =====================================================
# FEATURE ENGINEERING
# =====================================================

df['Profit Margin'] = np.where(
    df['Sales'] != 0,
    (df['Profit'] / df['Sales']) * 100,
    0
)

df['Order Year'] = df['Order Date'].dt.year

df['Order Month'] = df['Order Date'].dt.month

df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

# Shipping Duration

df['Shipping Days'] = (
    df['Ship Date'] - df['Order Date']
).dt.days

print("\nNew Features Created:")
display(
    df[
        [
            'Sales',
            'Profit',
            'Profit Margin',
            'Order Year',
            'Order Month',
            'Shipping Days'
        ]
    ].head()
)


New Features Created:


,Sales,Profit,Profit Margin,Order Year,Order Month,Shipping Days
0,261.9600,41.9136,16.00,2016,11,3
1,731.9400,219.5820,30.00,2016,11,3
2,14.6200,6.8714,47.00,2016,6,4
3,957.5775,-383.0310,-40.00,2015,10,7
4,22.3680,2.5164,11.25,2015,10,7


# Data Preparation Summary

- The dataset was loaded successfully.
- Duplicate records were identified and removed.
- Date columns were converted into datetime format.
- New business metrics were created:
  - Profit Margin
  - Order Year
  - Order Month
  - Shipping Days
- The dataset is now ready for exploratory and diagnostic analysis.

# Monthly Sales Trend Analysis

This section analyzes how sales evolved over time.

The objective is to identify:

- Overall sales trends
- Seasonal patterns
- Year-over-year growth
- Product category performance

An interactive widget allows the user to explore sales trends by product category.

In [10]:
# =====================================================
# MONTHLY SALES PREPARATION
# =====================================================

monthly_sales = (
    df.groupby(['Order Month-Year', 'Category'])['Sales']
      .sum()
      .reset_index()
)

monthly_sales['Date'] = (
    monthly_sales['Order Month-Year']
    .dt.to_timestamp()
)

monthly_sales.head()

,Order Month-Year,Category,Sales,Date
0,2014-01,Furniture,6242.525,2014-01-01
1,2014-01,Office Supplies,4851.080,2014-01-01
2,2014-01,Technology,3143.290,2014-01-01
3,2014-02,Furniture,1839.658,2014-02-01
4,2014-02,Office Supplies,1071.724,2014-02-01


In [11]:
# =====================================================
# INTERACTIVE SALES TREND ANALYSIS
# =====================================================

def plot_monthly_sales(category='All'):

    plt.figure(figsize=(14,6))

    if category == 'All':

        total_monthly = (
            df.groupby('Order Month-Year')['Sales']
              .sum()
        )

        plt.plot(
            total_monthly.index.to_timestamp(),
            total_monthly.values,
            marker='o',
            linewidth=2
        )

        plt.title(
            'Monthly Sales Trend - All Categories',
            fontsize=16,
            fontweight='bold'
        )

    else:

        category_data = monthly_sales[
            monthly_sales['Category'] == category
        ]

        plt.plot(
            category_data['Date'],
            category_data['Sales'],
            marker='o',
            linewidth=2
        )

        plt.title(
            f'Monthly Sales Trend - {category}',
            fontsize=16,
            fontweight='bold'
        )

    plt.xlabel("Date")
    plt.ylabel("Sales ($)")
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


categories = ['All'] + list(df['Category'].unique())

interact(
    plot_monthly_sales,
    category=Dropdown(
        options=categories,
        value='All',
        description='Category'
    )
)

interactive(children=(Dropdown(description='Category', options=('All', 'Furniture', 'Office Supplies', 'Techno…

<function __main__.plot_monthly_sales(category='All')>

In [12]:
# =====================================================
# SALES TREND KPIs
# =====================================================

monthly_total = (
    df.groupby('Order Month-Year')['Sales']
      .sum()
)

print("Highest Sales Month:")
print(monthly_total.idxmax())
print(f"${monthly_total.max():,.0f}")

print("\nLowest Sales Month:")
print(monthly_total.idxmin())
print(f"${monthly_total.min():,.0f}")

print("\nAverage Monthly Sales:")
print(f"${monthly_total.mean():,.0f}")

Highest Sales Month:
2017-11
$118,448

Lowest Sales Month:
2014-02
$4,520

Average Monthly Sales:
$47,858


# Key Findings

### Trend Analysis

- Sales show an overall upward/downward trend over time.
- Several sales peaks can be observed throughout the period.
- Certain months consistently outperform others, suggesting seasonality.

### Category Analysis

- Technology appears to generate the strongest sales performance.
- Furniture shows moderate growth.
- Office Supplies provide stable but lower sales volumes.

### Business Implications

The company should investigate the drivers behind peak sales periods and leverage these insights for future marketing campaigns.

In [13]:
monthly_total.sort_values(ascending=False).head()

,Sales
Order Month-Year,
2017-11,118447.8250
2016-12,96999.0430
2017-09,87866.6520
2017-12,83829.3188
2014-09,81777.3508


# Key Findings

### Trend Analysis

Monthly sales exhibit significant fluctuations throughout the observed period, with several strong peaks occurring during the second half of the year.

The highest sales month was November 2017, generating approximately $118,448 in revenue. Other top-performing periods include December 2016 and September 2017.

### Seasonality

The concentration of top sales months in September, November, and December suggests the presence of seasonal purchasing patterns. These peaks may be influenced by back-to-school demand, holiday shopping, and end-of-year purchasing activities.

### Business Implications

The company should increase inventory levels and marketing efforts before high-demand periods, particularly during the final quarter of the year.

Understanding the drivers behind these seasonal peaks can help optimize promotional campaigns and resource allocation.

In [14]:
yearly_sales = (
    df.groupby('Order Year')['Sales']
      .sum()
      .sort_index()
)

print(yearly_sales)

Order Year
2014    484247.4981
2015    470532.5090
2016    609205.5980
2017    733215.2552
Name: Sales, dtype: float64


In [15]:
growth = yearly_sales.pct_change() * 100

print(growth)

Order Year
2014          NaN
2015    -2.832227
2016    29.471521
2017    20.355962
Name: Sales, dtype: float64


Sales increased by X% between 2016 and 2017.

In [16]:
state_sales = (
    df.groupby('State')['Sales']
      .sum()
      .sort_values()
)

print(state_sales.tail(10))

State
Virginia         70636.7200
Michigan         76269.6140
Ohio             78258.1360
Illinois         80166.1010
Florida          89473.7080
Pennsylvania    116511.9140
Washington      138641.2700
Texas           170188.0458
New York        310876.2710
California      457687.6315
Name: Sales, dtype: float64


In [17]:
state_sales = (
    df.groupby('State')['Sales']
      .sum()
      .sort_values(ascending=True)
)

def plot_top_states(top_n=10):

    plt.figure(figsize=(12,6))

    top_states = state_sales.tail(top_n)

    plt.barh(
        top_states.index,
        top_states.values
    )

    plt.xlabel('Sales ($)')
    plt.ylabel('State')
    plt.title(
        f'Top {top_n} States by Sales'
    )

    plt.show()

interact(
    plot_top_states,
    top_n=(5,25,1)
)

interactive(children=(IntSlider(value=10, description='top_n', max=25, min=5), Output()), _dom_classes=('widge…

<function __main__.plot_top_states(top_n=10)>

# Geographic Sales Performance Analysis

### Key Findings

Sales are highly concentrated in a small number of states.

California is the strongest market, generating approximately $457,688 in sales, followed by New York with $310,876. Together, these two states account for a substantial portion of total company revenue.

Texas, Washington, and Pennsylvania also contribute significantly, but remain considerably behind the two leading states.

### Geographic Concentration

The distribution of sales is not uniform across the country. Revenue generation is concentrated in a limited number of high-performing states, suggesting that business performance depends heavily on specific geographic markets.

### Business Implications

Maintaining strong market presence in California and New York should remain a strategic priority.

At the same time, underperforming states may represent growth opportunities through targeted marketing campaigns, localized promotions, and improved product positioning.

In [18]:
top5_share = (
    state_sales.tail(5).sum()
    / df['Sales'].sum()
) * 100

print(
    f"Top 5 states contribute {top5_share:.2f}% of total sales."
)

Top 5 states contribute 51.97% of total sales.


In [19]:
comparison = (
    df[df['State'].isin(
        ['California','New York']
    )]
    .groupby('State')
    [['Sales','Profit']]
    .sum()
)

print(comparison)

                  Sales      Profit
State                              
California  457687.6315  76381.3871
New York    310876.2710  74038.5486


In [20]:
product_profit = (
    df.groupby('Product Name')['Profit']
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print(product_profit)

Product Name
Canon imageCLASS 2200 Advanced Copier                                          25199.9280
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind     7753.0390
Hewlett Packard LaserJet 3310 Copier                                            6983.8836
Canon PC1060 Personal Laser Copier                                              4570.9347
HP Designjet T520 Inkjet Large Format Printer - 24" Color                       4094.9766
Ativa V4110MDD Micro-Cut Shredder                                               3772.9461
3D Systems Cube Printer, 2nd Generation, Magenta                                3717.9714
Plantronics Savi W720 Multi-Device Wireless Headset System                      3696.2820
Ibico EPK-21 Electric Binding System                                            3345.2823
Zebra ZM400 Thermal Label Printer                                               3343.5360
Name: Profit, dtype: float64


# Top 10 Most Profitable Products Analysis

### Key Findings

The Canon imageCLASS 2200 Advanced Copier is by far the most profitable product in the portfolio, generating approximately $25,200 in profit.

A significant profitability gap exists between the leading product and the rest of the top-performing products, indicating a strong concentration of profit among a limited number of high-value items.

The majority of the most profitable products belong to technology and office equipment categories, including printers, copiers, shredders, and communication devices.

### Business Implications

The company should prioritize inventory availability and marketing support for these high-profit products.

Special attention should be given to the Canon imageCLASS 2200 Advanced Copier, as it represents a major contributor to overall profitability.

Expanding sales of similar high-margin products could significantly improve company performance.

In [21]:
top10_profit = product_profit.sum()

total_profit = df['Profit'].sum()

profit_share = (
    top10_profit / total_profit
) * 100

print(
    f"Top 10 products generate {profit_share:.2f}% of total profit."
)

Top 10 products generate 23.21% of total profit.


In [22]:
top_categories = (
    df.groupby('Category')['Profit']
      .sum()
      .sort_values(ascending=False)
)

print(top_categories)

Category
Technology         145454.9481
Office Supplies    122490.8008
Furniture           18451.2728
Name: Profit, dtype: float64


In [23]:
top_categories = (
    df.groupby('Category')['Profit']
      .sum()
      .sort_values(ascending=False)
)

print(top_categories)

Category
Technology         145454.9481
Office Supplies    122490.8008
Furniture           18451.2728
Name: Profit, dtype: float64


Technology products dominate profitability, suggesting that future growth initiatives should prioritize this category.

In [24]:
high_discount = df[df['Discount'] > 0.2]

print(
    "Average profit for discounts >20%:",
    high_discount['Profit'].mean()
)

print(
    "Loss percentage:",
    (high_discount['Profit'] < 0).mean() * 100
)

Average profit for discounts >20%: -97.1830983488873
Loss percentage: 96.76956209619526


Average profit for discounts >20% = -97.18 $
Loss percentage = 96.77%

# Discount Strategy Analysis

### Key Findings

The analysis reveals a strong negative relationship between discount levels and profitability.

Transactions receiving discounts greater than 20% generate an average loss of approximately $97 per order.

Furthermore, 96.8% of transactions with discounts above 20% result in negative profit.

### Business Implications

The current discount strategy appears unsustainable at higher discount levels.

While discounts may stimulate sales volume, excessive discounting significantly erodes profitability and creates financial losses.

### Recommendation

Management should consider implementing a discount ceiling of 20%.

Any promotional campaign exceeding this threshold should undergo additional profitability analysis before implementation.

Alternative strategies such as targeted promotions, loyalty programs, and product bundling may provide better financial outcomes than aggressive discounting.

In [25]:
for category in df['Category'].unique():

    cat = df[
        (df['Category'] == category)
        &
        (df['Discount'] > 0.2)
    ]

    print(
        category,
        "Average Profit:",
        round(cat['Profit'].mean(),2)
    )

Furniture Average Profit: -100.51
Office Supplies Average Profit: -69.32
Technology Average Profit: -197.42


trends varying by cathegory

In [26]:
print("Total Sales:", df['Sales'].sum())
print("Total Profit:", df['Profit'].sum())

print("\nProfit by Category:")
print(
    df.groupby('Category')['Profit']
      .sum()
      .sort_values(ascending=False)
)

Total Sales: 2297200.8603000003
Total Profit: 286397.0217

Profit by Category:
Category
Technology         145454.9481
Office Supplies    122490.8008
Furniture           18451.2728
Name: Profit, dtype: float64


# Executive Summary

## Business Performance Overview

The company generated total sales of approximately $2.30 million and total profit of $286,397, resulting in an overall profit margin of approximately 12.5%.

## Geographic Performance

Sales performance is highly concentrated in a limited number of states. California is the strongest market, generating approximately $457,688 in sales, followed by New York with $310,876. These states represent strategic markets that should remain a priority for future growth initiatives.

## Product and Category Performance

Technology is the most profitable category, generating over $145,000 in profit. Several high-margin products, particularly the Canon imageCLASS 2200 Advanced Copier, contribute disproportionately to company profitability.

## Discount Strategy Findings

The analysis revealed a strong negative relationship between discount levels and profitability. Transactions receiving discounts greater than 20% generate an average loss of approximately $97 per order, and 96.8% of such transactions result in negative profit.

## Strategic Opportunity

The company has significant opportunities to improve profitability by focusing on high-performing states, expanding sales of high-margin technology products, and reducing excessive discounting practices.


# Strategic Recommendations

### 1. Strengthen Presence in High-Performing States

Increase marketing investments and customer retention efforts in California and New York, as these states generate a substantial share of total revenue.

### 2. Prioritize Technology Products

Allocate additional resources to the Technology category, which delivers the highest profitability and contains several top-performing products.

### 3. Review Discount Policies

Implement a recommended discount ceiling of 20%. Promotions exceeding this threshold should require profitability validation before approval.

### 4. Improve Furniture Profitability

Furniture generates significantly lower profit than other categories. Pricing strategies, supplier costs, and discount practices should be reviewed to improve margins.

### 5. Develop Profitability Monitoring

Establish ongoing KPI monitoring through dashboards tracking sales, profit margin, geographic performance, and discount effectiveness.
